# Strategy comparison

Compare any registered long-only S&P 500 ETF/cash strategies on one interactive chart. The notebook imports the tested package; it does not duplicate strategy implementation.


In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from retail_sp500.backtest import BacktestConfig, run_many
from retail_sp500.data import enrich_with_risk_free_rate, load_shiller_data, synthetic_market_data
from retail_sp500.plotting import comparison_figure
from retail_sp500.strategies import select_strategies

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Load data

Keep `USE_SYNTHETIC = True` for an offline smoke test. Set it to `False` for the Shiller history. `INCLUDE_FRED = True` enables fractional Kelly by joining FRED TB3MS.


In [ ]:
USE_SYNTHETIC = True
INCLUDE_FRED = False

market = synthetic_market_data(periods=600) if USE_SYNTHETIC else load_shiller_data()
if INCLUDE_FRED and not USE_SYNTHETIC:
    market = enrich_with_risk_free_rate(market)

market.tail()


## Select strategies

Edit the keys below. Unsupported strategies are reported instead of receiving fabricated inputs.


In [ ]:
strategy_keys = [
    "buy-hold",
    "trend-10m",
    "vol-target-12",
    "trend-vol-12",
    "fractional-kelly",
]

strategies, skipped = select_strategies(
    strategy_keys,
    market,
    skip_unavailable=True,
)

skipped


In [ ]:
config = BacktestConfig(
    initial_cash=100_000,
    monthly_contribution=1_000,
    cash_annual_return=0.0,
)

results = run_many(market, strategies, config)


## Metrics and graph


In [ ]:
metrics = pd.DataFrame(
    {name: result.metrics for name, result in results.items()}
).T

metrics.sort_values("ending_value", ascending=False)


In [ ]:
comparison_figure(results).show()


## Inspect one strategy


In [ ]:
selected_name = next(iter(results))
results[selected_name].history.tail(24)
